# Build Dataset Validation

This notebook validates outputs from `src/data/build_dataset.py`.

Checks include:
- Train/val/test shapes and proportions
- Date boundary ordering (no temporal leakage)
- Overlap checks between splits
- Basic schema/target checks

In [6]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()
project_root = cwd if cwd.name == 'xai-retail-replenishment' else cwd.parent
processed_dir = project_root / 'data' / 'processed'

print('Project root :', project_root)
print('Processed dir:', processed_dir)

Project root : C:\Users\amrok\Desktop\Thesis\Project_XAI\xai-retail-replenishment
Processed dir: C:\Users\amrok\Desktop\Thesis\Project_XAI\xai-retail-replenishment\data\processed


In [7]:
train = pd.read_csv(processed_dir / 'train.csv')
val = pd.read_csv(processed_dir / 'val.csv')
test = pd.read_csv(processed_dir / 'test.csv')
full = pd.read_csv(processed_dir / 'full_merged.csv')

print('train shape:', train.shape)
print('val shape  :', val.shape)
print('test shape :', test.shape)
print('full shape :', full.shape)

train shape: (551808, 27)
val shape  : (68976, 27)
test shape : (68976, 27)
full shape : (2748981, 27)


In [8]:
def get_date_col(df: pd.DataFrame) -> str | None:
    for c in ['Date', 'date']:
        if c in df.columns:
            return c
    return None

train_date_col = get_date_col(train)
val_date_col = get_date_col(val)
test_date_col = get_date_col(test)

print('Date columns:', train_date_col, val_date_col, test_date_col)

Date columns: date date date


In [9]:
if train_date_col and val_date_col and test_date_col:
    train_dates = pd.to_datetime(train[train_date_col], errors='coerce').dropna()
    val_dates = pd.to_datetime(val[val_date_col], errors='coerce').dropna()
    test_dates = pd.to_datetime(test[test_date_col], errors='coerce').dropna()

    print('Train date range:', train_dates.min(), '->', train_dates.max())
    print('Val date range  :', val_dates.min(), '->', val_dates.max())
    print('Test date range :', test_dates.min(), '->', test_dates.max())

    assert train_dates.max() < val_dates.min(), 'Leakage: train extends into val period'
    assert val_dates.max() < test_dates.min(), 'Leakage: val extends into test period'
    print('Temporal split boundary check: PASS')
else:
    print('No date column found; temporal check skipped.')

Train date range: 2015-01-01 00:00:00 -> 2016-01-19 00:00:00
Val date range  : 2016-01-20 00:00:00 -> 2016-03-07 00:00:00
Test date range : 2016-03-08 00:00:00 -> 2016-04-24 00:00:00
Temporal split boundary check: PASS


In [10]:
# Date overlap checks between splits
if train_date_col and val_date_col and test_date_col:
    tr_set = set(pd.to_datetime(train[train_date_col], errors='coerce').dropna().unique())
    va_set = set(pd.to_datetime(val[val_date_col], errors='coerce').dropna().unique())
    te_set = set(pd.to_datetime(test[test_date_col], errors='coerce').dropna().unique())

    print('train ∩ val overlap dates :', len(tr_set & va_set))
    print('val ∩ test overlap dates  :', len(va_set & te_set))
    print('train ∩ test overlap dates:', len(tr_set & te_set))

    assert len(tr_set & va_set) == 0
    assert len(va_set & te_set) == 0
    assert len(tr_set & te_set) == 0
    print('Date overlap check: PASS')

train ∩ val overlap dates : 0
val ∩ test overlap dates  : 0
train ∩ test overlap dates: 0
Date overlap check: PASS


## Expected Outcome
- All assertions pass
- Temporal boundaries are strictly ordered
- No date overlap between train/val/test
- Row counts sum back to full dataset